# 03 — ASR bằng faster-whisper
#
Notebook chạy ASR cho **toàn bộ video raw**.
#
- Raw video là input bắt buộc.
- Scene manifest là input tùy chọn. Có scene manifest thì notebook tạo thêm
  `scene_asr.jsonl`.
- Video không có audio được ghi `status=no_audio`, không làm hỏng toàn batch.
- Model được load một lần rồi xử lý lần lượt các video.
- Mỗi tác vụ được bọc trong `Stage(...)` để in tiến độ + thời gian chạy,
  và có bảng tổng kết thời gian ở cuối.
#
Output tách riêng, không ghi đè metadata của notebook khác.

In [1]:
%pip install -q --no-cache-dir "faster-whisper>=1.1.1,<2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 113.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 101.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 107.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


## Stage tracker — tiến độ + thời gian cho từng tác vụ

In [2]:
import time
from contextlib import contextmanager
from typing import Any

_STAGE_LOG: list[dict[str, Any]] = []

@contextmanager
def Stage(name: str):
    """In trạng thái bắt đầu / kết thúc và lưu lại thời gian chạy của tác vụ."""
    t0 = time.perf_counter()
    print(f"▶ [{name}] bắt đầu...", flush=True)
    try:
        yield
    finally:
        dt = time.perf_counter() - t0
        _STAGE_LOG.append({"stage": name, "seconds": round(dt, 3)})
        print(f"✔ [{name}] xong sau {dt:.2f}s", flush=True)

def print_stage_summary():
    print("\n===== TỔNG KẾT THỜI GIAN =====")
    total = 0.0
    for row in _STAGE_LOG:
        print(f"  {row['stage']:<38} {row['seconds']:>8.2f}s")
        total += row["seconds"]
    print(f"  {'TỔNG CỘNG':<38} {total:>8.2f}s")
    print("===============================\n")

## Cấu hình

In [3]:
from pathlib import Path
import os

with Stage("Đọc cấu hình"):
    RAW_VIDEO_ROOT = Path(os.environ.get(
        "AIC_RAW_VIDEO_ROOT",
        "/kaggle/input/datasets/trongnhantran25/aic-nam-thang-ay/Videos_L21_a"
    ))
    SCENE_MANIFEST_PATH = Path(os.environ.get(
        "AIC_SCENE_MANIFEST",
        ""
    )) if os.environ.get("AIC_SCENE_MANIFEST", "") else None
    OUTPUT_ROOT = Path(os.environ.get(
        "AIC_STAGE03_OUTPUT",
        "/kaggle/working/aic_stage_03_asr"
    ))

    MODEL_NAME = "large-v3"
    MODEL_CACHE = "/kaggle/working/faster_whisper_cache"
    DEVICE = "cuda"
    DEVICE_INDEX = 0
    COMPUTE_TYPE_CANDIDATES = ["int8_float16", "float16"]
    LANGUAGE = None       # None = tự nhận diện
    BEAM_SIZE = 5
    VAD_FILTER = True
    MIN_SILENCE_MS = 500
    WORD_TIMESTAMPS = False
    CONDITION_ON_PREVIOUS_TEXT = False
    MAX_VIDEOS = 0
    PACK_VERSION = "1.1.0"

    print("  RAW_VIDEO_ROOT =", RAW_VIDEO_ROOT)
    print("  SCENE_MANIFEST =", SCENE_MANIFEST_PATH)
    print("  OUTPUT_ROOT    =", OUTPUT_ROOT)

▶ [Đọc cấu hình] bắt đầu...
  RAW_VIDEO_ROOT = /kaggle/input/datasets/trongnhantran25/aic-nam-thang-ay/Videos_L21_a
  SCENE_MANIFEST = None
  OUTPUT_ROOT    = /kaggle/working/aic_stage_03_asr
✔ [Đọc cấu hình] xong sau 0.00s


## Hàm I/O, discover video và kiểm tra audio

In [4]:
import hashlib
import json
import subprocess
import zipfile
from collections import defaultdict
from datetime import datetime, timezone

from faster_whisper import WhisperModel
from tqdm.auto import tqdm

VIDEO_EXTENSIONS = {".mp4", ".mkv", ".mov", ".avi", ".webm", ".m4v"}

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def atomic_write_bytes(path: Path, data: bytes) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + ".tmp")
    with temp.open("wb") as handle:
        handle.write(data)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temp, path)

def atomic_write_json(path: Path, payload: Any) -> None:
    atomic_write_bytes(
        path,
        (json.dumps(payload, ensure_ascii=False, indent=2) + "\n").encode("utf-8"),
    )

def atomic_write_jsonl(path: Path, rows: list[dict[str, Any]]) -> None:
    content = "".join(
        json.dumps(row, ensure_ascii=False, separators=(",", ":")) + "\n"
        for row in rows
    )
    atomic_write_bytes(path, content.encode("utf-8"))

def read_jsonl(path: Path) -> list[dict[str, Any]]:
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))
    return rows

def create_zip(folder: Path, zip_path: Path) -> Path:
    temp = zip_path.with_name(zip_path.name + ".tmp")
    files = sorted(p for p in folder.rglob("*") if p.is_file())
    with zipfile.ZipFile(temp, "w", zipfile.ZIP_DEFLATED, compresslevel=4) as archive:
        for path in tqdm(files, desc="  đóng gói zip", unit="file"):
            archive.write(path, path.relative_to(folder))
    os.replace(temp, zip_path)
    return zip_path

def discover_videos(root: Path) -> list[Path]:
    paths = sorted(
        path for path in root.rglob("*")
        if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS
    )
    if not paths:
        raise FileNotFoundError(f"No raw videos under {root}")
    return paths[:MAX_VIDEOS] if MAX_VIDEOS > 0 else paths

def has_audio(path: Path) -> bool:
    completed = subprocess.run(
        [
            "ffprobe", "-v", "error",
            "-select_streams", "a",
            "-show_entries", "stream=index",
            "-of", "csv=p=0",
            str(path),
        ],
        check=True,
        capture_output=True,
        text=True,
    )
    return bool(completed.stdout.strip())

def project_segments_to_scenes(
    video_id: str,
    segments: list[dict[str, Any]],
    scenes: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    rows = []
    for scene in scenes:
        start = float(scene["start_sec"])
        end = float(scene["end_sec"])
        ids = []
        texts = []
        for segment in segments:
            overlap = min(end, float(segment["end_sec"])) - max(
                start, float(segment["start_sec"])
            )
            if overlap <= 0:
                continue
            ids.append(segment["asr_segment_id"])
            if segment["text"]:
                texts.append(segment["text"])
        text = " ".join(texts).strip()
        rows.append({
            "video_id": video_id,
            "scene_id": scene["scene_id"],
            "status": "generated" if text else "empty_speech",
            "text": text,
            "asr_segment_ids": ids,
        })
    return rows

## Chuẩn bị thư mục output + phát hiện video + kiểm tra audio

In [5]:
started = time.perf_counter()

with Stage("Chuẩn bị thư mục output"):
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    (OUTPUT_ROOT / "manifests").mkdir(parents=True, exist_ok=True)

with Stage("Phát hiện video + đọc scene manifest"):
    video_paths = discover_videos(RAW_VIDEO_ROOT)
    print(f"  tìm thấy {len(video_paths)} video (giới hạn MAX_VIDEOS={MAX_VIDEOS})")

    scene_rows = (
        read_jsonl(SCENE_MANIFEST_PATH)
        if SCENE_MANIFEST_PATH and SCENE_MANIFEST_PATH.exists()
        else []
    )
    scenes_by_video: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for scene in scene_rows:
        scenes_by_video[str(scene["video_id"])].append(scene)
    print(f"  scene manifest: {len(scene_rows)} scene, {len(scenes_by_video)} video có scene")

with Stage("Kiểm tra audio (ffprobe) cho từng video"):
    audio_flags = {}
    for path in tqdm(video_paths, desc="  ffprobe", unit="video"):
        audio_flags[path.stem] = has_audio(path)
    n_with_audio = sum(audio_flags.values())
    print(f"  {n_with_audio}/{len(video_paths)} video có audio")

▶ [Chuẩn bị thư mục output] bắt đầu...
✔ [Chuẩn bị thư mục output] xong sau 0.00s
▶ [Phát hiện video + đọc scene manifest] bắt đầu...
  tìm thấy 29 video (giới hạn MAX_VIDEOS=0)
  scene manifest: 0 scene, 0 video có scene
✔ [Phát hiện video + đọc scene manifest] xong sau 0.02s
▶ [Kiểm tra audio (ffprobe) cho từng video] bắt đầu...


  ffprobe:   0%|          | 0/29 [00:00<?, ?video/s]

  29/29 video có audio
✔ [Kiểm tra audio (ffprobe) cho từng video] xong sau 5.45s


## Load model rõ ràng theo compute type

In [6]:
with Stage("Load faster-whisper model"):
    model = None
    selected_compute_type = ""
    model_errors = []

    if any(audio_flags.values()):
        for compute_type in COMPUTE_TYPE_CANDIDATES:
            print(f"  thử compute_type={compute_type} ...")
            t0 = time.perf_counter()
            try:
                model = WhisperModel(
                    MODEL_NAME,
                    device=DEVICE,
                    device_index=DEVICE_INDEX,
                    compute_type=compute_type,
                    download_root=MODEL_CACHE,
                )
                selected_compute_type = compute_type
                print(f"  ✔ load thành công compute_type={compute_type} trong {time.perf_counter()-t0:.2f}s")
                break
            except Exception as exc:
                model_errors.append({"compute_type": compute_type, "error": repr(exc)})
                print(f"  ✘ [MODEL LOAD FAILED] {compute_type}: {exc!r}")
        if model is None:
            raise RuntimeError(f"Cannot load faster-whisper: {model_errors}")
    else:
        print("  không video nào có audio -> bỏ qua load model")

▶ [Load faster-whisper model] bắt đầu...
  thử compute_type=int8_float16 ...


  ✔ load thành công compute_type=int8_float16 trong 17.81s
✔ [Load faster-whisper model] xong sau 17.81s


## Chạy ASR cho từng video (tiến độ theo video + theo segment)

In [7]:
global_segments: list[dict[str, Any]] = []
global_scene_asr: list[dict[str, Any]] = []
video_status_rows: list[dict[str, Any]] = []

with Stage(f"Chạy ASR cho {len(video_paths)} video"):
    for video_number, video_path in enumerate(
        tqdm(video_paths, desc="  tiến độ video", unit="video"), start=1
    ):
        video_id = video_path.stem
        print(f"\n  [{video_number}/{len(video_paths)}] {video_id}")
        video_output = OUTPUT_ROOT / "videos" / video_id
        video_output.mkdir(parents=True, exist_ok=True)
        t_video = time.perf_counter()

        if not audio_flags[video_id]:
            segments = []
            language = ""
            language_probability = 0.0
            status = "no_audio"
            print("    không có audio -> status=no_audio")
        else:
            generator, info = model.transcribe(
                str(video_path),
                language=LANGUAGE,
                beam_size=BEAM_SIZE,
                vad_filter=VAD_FILTER,
                vad_parameters={"min_silence_duration_ms": MIN_SILENCE_MS},
                word_timestamps=WORD_TIMESTAMPS,
                condition_on_previous_text=CONDITION_ON_PREVIOUS_TEXT,
            )
            segments = []
            for index, segment in enumerate(tqdm(generator, desc=f"    segment {video_id}", unit="seg", leave=False)):
                words = []
                if WORD_TIMESTAMPS and segment.words:
                    words = [
                        {
                            "word": str(word.word or "").strip(),
                            "start_sec": float(word.start or 0.0),
                            "end_sec": float(word.end or 0.0),
                            "probability": float(word.probability or 0.0),
                        }
                        for word in segment.words
                    ]
                segments.append({
                    "video_id": video_id,
                    "asr_segment_id": f"{video_id}_A{index:06d}",
                    "start_sec": float(segment.start),
                    "end_sec": float(segment.end),
                    "text": str(segment.text or "").strip(),
                    "avg_logprob": float(segment.avg_logprob or 0.0),
                    "no_speech_prob": float(segment.no_speech_prob or 0.0),
                    "compression_ratio": float(segment.compression_ratio or 0.0),
                    "words": words,
                    "model": f"faster-whisper:{MODEL_NAME}",
                    "compute_type": selected_compute_type,
                })
            language = str(info.language or "")
            language_probability = float(info.language_probability or 0.0)
            status = "generated"

        scene_asr = project_segments_to_scenes(
            video_id,
            segments,
            scenes_by_video.get(video_id, []),
        )

        atomic_write_jsonl(video_output / "asr_segments.jsonl", segments)
        atomic_write_jsonl(video_output / "scene_asr.jsonl", scene_asr)
        atomic_write_json(
            video_output / "asr_info.json",
            {
                "video_id": video_id,
                "status": status,
                "model": f"faster-whisper:{MODEL_NAME}" if status != "no_audio" else "",
                "compute_type": selected_compute_type if status != "no_audio" else "",
                "language": language,
                "language_probability": language_probability,
                "segment_count": len(segments),
                "scene_projection_count": len(scene_asr),
            },
        )
        atomic_write_json(
            video_output / "_SUCCESS.json",
            {
                "status": "success",
                "video_id": video_id,
                "asr_status": status,
                "segment_count": len(segments),
            },
        )

        global_segments.extend(segments)
        global_scene_asr.extend(scene_asr)
        video_status_rows.append({
            "video_id": video_id,
            "status": status,
            "segment_count": len(segments),
            "scene_projection_count": len(scene_asr),
            "language": language,
            "language_probability": language_probability,
        })
        print(f"    xong trong {time.perf_counter()-t_video:.2f}s "
              f"({len(segments)} segment, {len(scene_asr)} scene_asr)")

▶ [Chạy ASR cho 29 video] bắt đầu...


  tiến độ video:   0%|          | 0/29 [00:00<?, ?video/s]


  [1/29] L21_V001


    segment L21_V001: 0seg [00:00, ?seg/s]

    xong trong 176.95s (264 segment, 0 scene_asr)

  [2/29] L21_V002


...

    xong trong 155.03s (241 segment, 0 scene_asr)

  [28/29] L21_V030


    segment L21_V030: 0seg [00:00, ?seg/s]

    xong trong 152.39s (199 segment, 0 scene_asr)

  [29/29] L21_V031


    segment L21_V031: 0seg [00:00, ?seg/s]

    xong trong 145.44s (213 segment, 0 scene_asr)
✔ [Chạy ASR cho 29 video] xong sau 4398.40s


## Ghi manifest tổng hợp + đóng gói

In [8]:
with Stage("Ghi manifest tổng hợp"):
    atomic_write_jsonl(OUTPUT_ROOT / "manifests" / "asr_segments.jsonl", global_segments)
    atomic_write_jsonl(OUTPUT_ROOT / "manifests" / "scene_asr.jsonl", global_scene_asr)
    atomic_write_jsonl(OUTPUT_ROOT / "manifests" / "asr_video_status.jsonl", video_status_rows)
    atomic_write_json(
        OUTPUT_ROOT / "model_info.json",
        {
            "component": "asr",
            "model": f"faster-whisper:{MODEL_NAME}",
            "selected_compute_type": selected_compute_type,
            "compute_type_attempts": COMPUTE_TYPE_CANDIDATES,
            "model_load_errors": model_errors,
            "beam_size": BEAM_SIZE,
            "vad_filter": VAD_FILTER,
            "word_timestamps": WORD_TIMESTAMPS,
            "condition_on_previous_text": CONDITION_ON_PREVIOUS_TEXT,
            "pack_version": PACK_VERSION,
        },
    )
    atomic_write_json(
        OUTPUT_ROOT / "_SUCCESS.json",
        {
            "status": "success",
            "stage": "03_asr",
            "video_count": len(video_paths),
            "segment_count": len(global_segments),
            "scene_projection_count": len(global_scene_asr),
            "runtime_sec": round(time.perf_counter() - started, 3),
            "created_at_utc": utc_now(),
        },
    )

with Stage("Đóng gói output thành zip"):
    ZIP_PATH = Path("/kaggle/working/03_asr_output.zip")
    create_zip(OUTPUT_ROOT, ZIP_PATH)

print({
    "videos": len(video_paths),
    "segments": len(global_segments),
    "scene_asr": len(global_scene_asr),
    "zip": str(ZIP_PATH),
})
print_stage_summary()

▶ [Ghi manifest tổng hợp] bắt đầu...
✔ [Ghi manifest tổng hợp] xong sau 0.10s
▶ [Đóng gói output thành zip] bắt đầu...


  đóng gói zip:   0%|          | 0/121 [00:00<?, ?file/s]

✔ [Đóng gói output thành zip] xong sau 0.13s
{'videos': 29, 'segments': 6446, 'scene_asr': 0, 'zip': '/kaggle/working/03_asr_output.zip'}

===== TỔNG KẾT THỜI GIAN =====
  Đọc cấu hình                               0.00s
  Chuẩn bị thư mục output                    0.00s
  Phát hiện video + đọc scene manifest       0.01s
  Kiểm tra audio (ffprobe) cho từng video     5.45s
  Load faster-whisper model                 17.81s
  Chạy ASR cho 29 video                   4398.40s
  Ghi manifest tổng hợp                      0.10s
  Đóng gói output thành zip                  0.12s
  TỔNG CỘNG                               4421.90s



## Kiểm tra timestamp và trạng thái video

In [9]:
with Stage("Validation"):
    for segment in global_segments:
        assert segment["start_sec"] <= segment["end_sec"]
        assert segment["video_id"]
        assert segment["asr_segment_id"]
    assert len(video_status_rows) == len(video_paths)
    print("  Validation passed:", len(video_status_rows), "videos")

▶ [Validation] bắt đầu...
  Validation passed: 29 videos
✔ [Validation] xong sau 0.01s
